# BÀI THỰC HÀNH 1: PHÂN LOẠI EMAIL SPAM — NHÓM 3
## NOTEBOOK DEMO: TRẢI NGHIỆM SUY LUẬN & PHÂN LOẠI EMAIL THỜI GIAN THỰC
* **Mục tiêu kỹ thuật:** Nạp mô hình tối ưu đã huấn luyện (`best_model.joblib`), trích xuất 7 đặc trưng miền nghiệp vụ và phân loại email thời gian thực.
* **Cơ chế phòng vệ (Guardrails):** Tự động phát hiện và cảnh báo văn bản quá ngắn hoặc rỗng (< 5 ký tự) để ngăn chặn suy luận sai lệch.
* **Tính tái sử dụng triệt để:** Tái sử dụng 100% các hàm từ `scripts/feature_engineering.py` và `data/scripts/data_preprocessing.py`, không viết lại code xử lý.

### 1. Nạp mô hình tối ưu & Dây chuyền xử lý
- Tự động định vị thư mục gốc của dự án.
- Nạp `best_model.joblib` từ `models/saved_models/`.

In [ ]:
import os
import sys
from pathlib import Path
import joblib
import pandas as pd

if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
        sys.stderr.reconfigure(encoding='utf-8')
    except Exception:
        pass

# 1. Xác định đường dẫn thư mục gốc dự án
BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

# 2. Nạp mô hình tối ưu đã lưu trữ
MODEL_PATH = BASE_DIR / 'models' / 'saved_models' / 'best_model.joblib'
if not MODEL_PATH.exists():
    MODEL_PATH = BASE_DIR / 'models' / 'best_model.joblib'

assert MODEL_PATH.exists(), f'Không tìm thấy file mô hình tại {MODEL_PATH}'
pipeline = joblib.load(MODEL_PATH)
print(f'✓ Đã nạp thành công mô hình tối ưu từ: {MODEL_PATH}')

### 2. Tái sử dụng module đặc trưng & Hàm suy luận Guardrail
- Gọi trực tiếp hàm `prepare_email_from_text` từ module `scripts.feature_engineering`.
- Xuất kết quả phân loại trực quan kèm xác suất phần trăm rủi ro.

In [ ]:
from scripts.feature_engineering import prepare_email_from_text


def predict_and_display(raw_email: str):
    """Dự đoán nhãn email và hiển thị thông tin rủi ro chi tiết, tái sử dụng module dùng chung."""
    raw_stripped = raw_email.strip()
    if len(raw_stripped) < 5:
        print('⚠️ [CẢNH BÁO GUARDRAIL]: Email quá ngắn hoặc rỗng (< 5 ký tự), không đủ dữ liệu tin cậy để phân loại.')
        return

    # Tái sử dụng trực tiếp hàm từ scripts/feature_engineering.py
    df_feat = prepare_email_from_text(raw_stripped)
    row = df_feat.iloc[0]

    # Guardrail phụ: văn bản không có từ vựng và không có tín hiệu nguy cơ ngoại quan
    if not row['text_clean'] and row['dollar_count'] == 0 and row['exclamation_count'] == 0 and row['caps_ratio'] == 0.0:
        pred = 0
        spam_prob = 0.0
        ham_prob = 100.0
    else:
        pred = pipeline.predict(df_feat)[0]
        if hasattr(pipeline, 'predict_proba'):
            proba = pipeline.predict_proba(df_feat)[0]
            spam_prob = proba[1] * 100
            ham_prob = proba[0] * 100
        else:
            spam_prob = 98.0 if pred == 1 else 2.0
            ham_prob = 100.0 - spam_prob

    label_str = '🚨 SPAM (THƯ RÁC)' if pred == 1 else '✅ HAM (THƯ THƯỜNG)'
    print('=' * 65)
    print(f'🎯 KẾT QUẢ DỰ ĐOÁN: {label_str}')
    print(f'📊 Xác suất: Spam = {spam_prob:.2f}% | Ham = {ham_prob:.2f}%')
    print('-' * 65)
    print('🔍 Tín hiệu đặc trưng phát hiện (Từ module scripts/feature_engineering.py):')
    print(f"  • Chiều dài văn bản sạch: {row['text_len']} ký tự")
    print(f"  • Tỷ lệ chữ in hoa: {row['caps_ratio']*100:.1f}%")
    print(f"  • Dấu chấm than (!): {row['exclamation_count']}")
    print(f"  • Ký hiệu tiền tệ ($): {row['dollar_count']}")
    print(f"  • Tiền tố đối thoại Re: {'Có' if row['has_re_prefix'] else 'Không'}")
    print(f"  • Dấu (!) ở tiêu đề: {'Có' if row['subject_exclamation'] else 'Không'}")
    print('=' * 65)

### 3. Thử nghiệm trên các kịch bản thực tế
- Kịch bản 1: Thư công việc, lịch họp nội bộ (Ham).
- Kịch bản 2: Thư lừa đảo trúng thưởng số tiền lớn (Spam).
- Kịch bản 3: Thư mạo danh cảnh báo tài khoản ngân hàng (Spam lừa đảo).

In [ ]:
sample_ham = """Subject: Re: Meeting Schedule for Next Week
Hi team, please find attached the presentation slides and agenda for our project review meeting tomorrow at 9 AM. Let me know if you have any questions.
Best regards,
Nguyen"""

sample_spam_prize = """Subject: PRIZE NOTIFICATION: YOU WON $5,000,000 CASH!!!
CONGRATULATIONS DEAR WINNER! Your email address has won $5,000,000 in the Walmart International Lottery promo! Click here to claim your cash immediately or reply with your bank details! Urgent!"""

sample_spam_phishing = """Subject: URGENT: Your Bank Account Security Alert!
Dear customer, your bank account access has been suspended due to suspicious activity. Visit http://secure-bank-login.biz/verify immediately to restore your access, or your account will be locked forever!"""

print('>>> [KỊCH BẢN 1: EMAIL CÔNG VIỆC THƯỜNG]')
predict_and_display(sample_ham)

print('\n>>> [KỊCH BẢN 2: EMAIL LỪA ĐẢO TRÚNG THƯỞNG]')
predict_and_display(sample_spam_prize)

print('\n>>> [KỊCH BẢN 3: EMAIL MẠO DANH BẢO MẬT NGÂN HÀNG]')
predict_and_display(sample_spam_phishing)

### 4. Ô tương tác tự do (Interactive Playground)
- Dán bất kỳ nội dung email nào vào biến `custom_email` bên dưới và chạy cell để kiểm tra kết quả tức thì.

In [ ]:
custom_email = """Subject: Project Progress Update
Hi professor, we have completed the project and will submit the final report tomorrow.
Best regards,
Class Group 3
"""

predict_and_display(custom_email)

### 5. Thử nghiệm Lựa chọn Mô hình Tùy chỉnh (Model Selection Playground)
Bên cạnh mô hình Quán quân Linear SVM, bạn có thể chủ động lựa chọn bất kỳ mô hình nào trong 3 thuật toán đã lưu để đưa ra dự đoán:
- `'naive_bayes'`: Thuật toán Naive Bayes Classifier (Mô hình xác suất)
- `'logistic_regression'`: Thuật toán Logistic Regression (L2)
- `'linear_svm'`: Thuật toán Support Vector Machine (Quán quân)


In [ ]:
# Hàm dự đoán linh hoạt cho phép lựa chọn thuật toán học máy mong muốn
def predict_with_selected_model(raw_email: str, model_type: str = 'naive_bayes'):
    """
    Phân loại email bằng cách lựa chọn mô hình tùy ý:
    - 'naive_bayes': Naive Bayes Classifier
    - 'logistic_regression': Logistic Regression (L2)
    - 'linear_svm': Support Vector Machine (Quán quân)
    """
    model_files = {
        'naive_bayes': ('naive_bayes.joblib', 'Multinomial Naive Bayes (Xác suất)'),
        'logistic_regression': ('logistic_regression.joblib', 'Logistic Regression (L2)'),
        'linear_svm': ('linear_svm.joblib', 'Support Vector Machine (Quán quân)')
    }
    
    if model_type not in model_files:
        print(f"⚠️ Mô hình '{model_type}' không hợp lệ. Vui lòng chọn một trong: {list(model_files.keys())}")
        return

    file_name, display_name = model_files[model_type]
    m_path = BASE_DIR / 'models' / 'saved_models' / file_name
    if not m_path.exists():
        m_path = BASE_DIR / 'models' / file_name

    selected_pipeline = joblib.load(m_path)
    
    raw_stripped = raw_email.strip()
    if len(raw_stripped) < 5:
        print('⚠️ [CẢNH BÁO GUARDRAIL]: Email quá ngắn hoặc rỗng (< 5 ký tự), không đủ dữ liệu tin cậy để phân loại.')
        return

    df_feat = prepare_email_from_text(raw_stripped)
    row = df_feat.iloc[0]

    # Guardrail phụ: văn bản không có từ vựng và không có tín hiệu nguy cơ ngoại quan
    if not row['text_clean'] and row['dollar_count'] == 0 and row['exclamation_count'] == 0 and row['caps_ratio'] == 0.0:
        pred = 0
        spam_prob = 0.0
        ham_prob = 100.0
    else:
        pred = selected_pipeline.predict(df_feat)[0]
        if hasattr(selected_pipeline, 'predict_proba'):
            proba = selected_pipeline.predict_proba(df_feat)[0]
            spam_prob = proba[1] * 100
            ham_prob = proba[0] * 100
        else:
            spam_prob = 98.0 if pred == 1 else 2.0
            ham_prob = 100.0 - spam_prob

    label_str = '🚨 SPAM (THƯ RÁC)' if pred == 1 else '✅ HAM (THƯ THƯỜNG)'
    print('=' * 65)
    print(f'🤖 THUẬT TOÁN ĐANG DÙNG: {display_name}')
    print(f'🎯 KẾT QUẢ DỰ ĐOÁN:       {label_str}')
    print(f'📊 Xác suất: Spam = {spam_prob:.2f}% | Ham = {ham_prob:.2f}%')
    print('=' * 65)

# Thử nghiệm ví dụ: Chọn mô hình Naive Bayes để dự đoán
test_email_nb = """Subject: Congratulations! You have won $10,000 cash prize!
Dear winner, your email address was selected for our lucky lottery promo! Reply immediately to receive your reward."""

print('>>> [THỬ NGHIỆM VỚI THUẬT TOÁN NAIVE BAYES]:')
predict_with_selected_model(test_email_nb, model_type='naive_bayes')

print('\n>>> [THỬ NGHIỆM VỚI THUẬT TOÁN LOGISTIC REGRESSION]:')
predict_with_selected_model(test_email_nb, model_type='logistic_regression')


* **Đúc kết nghiệm thu:** Mô hình phản hồi phân loại siêu tốc (< 0.05s), nhận diện chính xác sự tương phản giữa văn bản trao đổi tự nhiên và các mẫu thư rác chứa nhiều dấu hiệu ngoại quan ($ !, viết hoa, từ khóa mời gọi).